# Sliding Window Attention from Scratch

This notebook is a teaching-first walkthrough of a transformer that uses **sliding window attention** to efficiently handle long sequences.
We build every component from scratch in PyTorch and explain the reasoning behind each design choice.

Instead of the standard O(n^2) attention that lets every token attend to every other token, we restrict most layers to a **local window** and reserve a few **global layers** for long-range communication.
This is the strategy used by Gemma 3, OLMo 3, and early Mistral models.

## Audience
- Learners who have already worked through the simple transformer and Llama notebooks in this series.
- Anyone who wants to understand how production models handle 128K+ context windows without quadratic cost.

## Prerequisites
- Comfortable with PyTorch tensors, modules, and training loops.
- Understand standard causal self-attention (queries, keys, values, masking).
- Familiar with the Llama building blocks: RMSNorm, RoPE, SwiGLU, and GQA. We will include brief recaps here, but the Llama notebook has the deep explanations.

## Learning Goals
By the end of this notebook, you should be able to:
1. Explain **why** standard attention is O(n^2) and why that matters for long contexts.
2. Describe how a **sliding window mask** restricts attention to a local neighborhood.
3. Understand the **alternating local/global pattern** and why both layer types are needed.
4. Explain what **QK-Norm** does and why it stabilizes training at scale.
5. Reason about the **effective receptive field** that grows with depth even when individual layers are local.
6. Build and train a sliding window transformer end to end.

## Outline

1. Set up the environment and imports.
2. Define `SlidingWindowConfig` with the new knobs: `window_size`, `global_every`, `use_qk_norm`.
3. Recap the shared building blocks from the Llama notebook: RMSNorm, RoPE, SwiGLU.
4. Build the sliding-window-specific components one at a time:
   - QKNorm,
   - SlidingWindowAttention (the key teaching moment),
   - Mask visualization,
   - SlidingWindowBlock.
5. Assemble the full `SlidingWindowModel` with its alternating local/global pattern.
6. Smoke-test the model.
7. Build the data pipeline and verify it.
8. Add training utilities and a full training loop.
9. Run a lightweight pipeline preview.
10. Add text generation.
11. Optional checkpoint demo.
12. Exercises.
13. Common pitfalls and next steps.

## 1. Setup

This install cell keeps the notebook easy to run in a fresh environment such as Colab.
If your environment already has these packages, rerunning this cell is harmless.

In [ ]:
!pip install torch numpy requests matplotlib -q

## 2. Imports

We gather the core tools once so every later cell can stay focused on one idea.

Teaching note:
- `dataclass` gives us a clean config object.
- `Path` helps keep file paths portable between local Jupyter and Colab-style environments.
- `torch.nn` holds reusable neural network layers.
- `torch.nn.functional` contains tensor-level math functions like `softmax` and `cross_entropy`.
- `matplotlib` is used only once for mask visualization.

In [ ]:
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuration

### The O(n^2) Problem

Before we define the config, let us understand the problem we are solving.

Standard self-attention computes a score between **every pair** of tokens. If the sequence has `n` tokens, that means `n * n` score computations. For short sequences this is fine, but modern models need context windows of 128K tokens or more. At that scale:

- 128K tokens means `128,000 * 128,000 = 16.4 billion` attention scores **per layer, per head**.
- The KV-cache (used during generation) stores key and value vectors for every past token, consuming enormous memory.

This quadratic cost is the single biggest bottleneck for long-context language models.

### The Sliding Window Solution

The insight: **most tokens only need local context**. When you read "the cat sat on the ___", the answer depends on the last few words, not on a paragraph from 10,000 tokens ago.

Sliding window attention restricts each token to attend only to a **local window** of `w` nearby tokens. This makes attention `O(n * w)` instead of `O(n^2)`. When `w = 64` and `n = 128,000`, that is a **2000x reduction** in work.

But some information genuinely needs to flow long-range (e.g., a system prompt at the beginning of a conversation). So modern models **alternate** between:
- **Local layers**: sliding window attention (cheap, captures local patterns like grammar and word relationships)
- **Global layers**: full attention (expensive, captures long-range dependencies like the system prompt)

The ratio varies by model:
- **Gemma 3** (Google, 2025): 5:1 local:global (aggressive -- mostly local)
- **OLMo 3** (AI2, 2025): 3:1 local:global
- **Mistral** (early versions): all sliding window, no global layers at all

### Effective Receptive Field

Even local-only layers can transmit information globally over multiple hops. Layer 1's window overlaps with Layer 2's window, so a token at position 0 can influence position 64 through Layer 1, and then position 128 through Layer 2. With `L` local layers and window size `w`, the effective receptive field is approximately `L * w` tokens.

### `SlidingWindowConfig`

This dataclass extends the familiar transformer config with three new fields:
- `window_size`: how many past tokens each local layer can see.
- `global_every`: insert one global (full attention) layer every N layers. With `global_every=3` and 6 layers, layers 2 and 5 are global (0-indexed), giving a 2:1 local:global ratio.
- `use_qk_norm`: whether to normalize Q and K before computing attention scores (explained in detail later).

In [ ]:
@dataclass
class SlidingWindowConfig:
    vocab_size: int = 256
    n_embd: int = 64
    n_head: int = 4
    n_kv_head: int = 2
    n_layer: int = 6            # more layers to show the alternating pattern
    seq_len: int = 256
    dropout: float = 0.0
    rope_theta: float = 10000.0

    # Sliding window specific
    window_size: int = 64       # local attention window
    global_every: int = 3       # insert a global attention layer every N layers
    use_qk_norm: bool = True    # normalize Q and K before attention

config = SlidingWindowConfig()
print(config)
print(f"\nWith {config.n_layer} layers and global_every={config.global_every}:")
for i in range(config.n_layer):
    is_global = ((i + 1) % config.global_every == 0)
    kind = "GLOBAL" if is_global else "local"
    print(f"  Layer {i}: {kind}")

## 4. Shared Building Blocks (Recap)

The sliding window model reuses three components from the Llama architecture. We include them here with brief recaps so this notebook is fully self-contained. If any of these feel unfamiliar, work through the Llama notebook first.

### `RMSNorm`

**Recap**: RMSNorm normalizes a vector by its root-mean-square magnitude, then rescales with a learnable weight.
It is simpler and slightly faster than LayerNorm because it skips the mean-subtraction step.
Every modern transformer uses it for pre-norm residual connections.

Formula: `RMSNorm(x) = x / sqrt(mean(x^2) + eps) * gamma`

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

### RoPE (Rotary Position Embeddings)

**Recap**: Instead of adding learned position vectors to token embeddings, RoPE encodes position by **rotating** query and key vectors. Each pair of dimensions gets rotated by an angle that depends on the token's position, so the dot product `Q . K` naturally captures **relative distance** between tokens.

Why this matters for sliding window attention: RoPE makes the model aware of how far apart two tokens are, which pairs perfectly with a window that restricts attention by distance.

Two functions:
- `precompute_rope_freqs`: builds the cosine and sine tables once.
- `apply_rope`: rotates Q and K using those tables.

In [ ]:
def precompute_rope_freqs(head_dim: int, seq_len: int, theta: float = 10000.0):
    """Precompute the complex exponentials for RoPE.

    Returns cos and sin tensors of shape (seq_len, head_dim // 2).
    """
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len).float()
    angles = torch.outer(t, freqs)  # (seq_len, head_dim // 2)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply rotary embeddings to a tensor.

    Args:
        x: shape (batch, n_head, seq_len, head_dim)
        cos, sin: shape (seq_len, head_dim // 2), precomputed frequencies
    """
    T = x.shape[2]
    x_pairs = x.float().reshape(*x.shape[:-1], -1, 2)
    x0 = x_pairs[..., 0]  # even dimensions
    x1 = x_pairs[..., 1]  # odd dimensions

    cos_t = cos[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim//2)
    sin_t = sin[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim//2)

    out0 = x0 * cos_t - x1 * sin_t
    out1 = x0 * sin_t + x1 * cos_t

    out = torch.stack([out0, out1], dim=-1).flatten(-2)
    return out.type_as(x)

### `SwiGLU`

**Recap**: SwiGLU replaces the simple ReLU/GELU MLP with a gated architecture:

`SwiGLU(x) = Swish(W_gate * x) * (W_up * x)`

The gate learns **what information to let through**, while the up-projection learns **what information to compute**. This separation consistently outperforms standard activations.

Because we have two up-projections (gate + up) instead of one, the hidden dimension is adjusted to `4 * n_embd * 2/3` to keep the total parameter count similar to a standard 4x-expansion MLP.

Note: `SwiGLU` needs a config with at least `n_embd` and `dropout`. We use a small helper dataclass so it works standalone.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_dim = int(4 * config.n_embd * 2 / 3)
        hidden_dim = ((hidden_dim + 7) // 8) * 8  # round to nearest multiple of 8

        self.w_gate = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.w_up = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.w_gate(x))  # Swish = SiLU = x * sigmoid(x)
        up = self.w_up(x)
        x = gate * up
        x = self.w_down(x)
        x = self.dropout(x)
        return x

## 5. Sliding-Window-Specific Components

This is where the new ideas live. We build three new classes:
1. **QKNorm** -- stabilizes attention scores by normalizing queries and keys.
2. **SlidingWindowAttention** -- the heart of this notebook, with mask construction for both global and local layers.
3. **SlidingWindowBlock** -- combines attention with SwiGLU in a pre-norm residual block.

### `QKNorm`

**The problem**: In standard attention, the dot product `Q . K` can grow very large, especially in deep or wide models. When the attention logits become huge, softmax produces near-one-hot distributions -- almost all the attention weight goes to a single token. This makes training unstable because gradients become very small for the ignored tokens and very large for the attended one.

**The solution**: Normalize Q and K to unit-ish length before computing their dot product. After normalization, `Q . K` is bounded and behaves like **cosine similarity** scaled by a learnable temperature (the RMSNorm weight acts as the temperature).

The formula becomes:
```
attention = softmax(Q_norm . K_norm / sqrt(d_k))
```

Where `Q_norm = RMSNorm(Q)` and `K_norm = RMSNorm(K)`.

This is a simple idea with a big practical impact. Gemma 3 and OLMo 3 both use it to stabilize training at scale.

**Implementation**: QKNorm is just two independent RMSNorm layers, one for Q and one for K. The normalization happens along the `head_dim` dimension, meaning each individual attention head's query and key vectors are normalized independently.

In [ ]:
class QKNorm(nn.Module):
    def __init__(self, head_dim: int):
        super().__init__()
        self.q_norm = RMSNorm(head_dim)
        self.k_norm = RMSNorm(head_dim)

    def forward(self, q: torch.Tensor, k: torch.Tensor):
        """Normalize Q and K along the head dimension.

        Args:
            q: (batch, n_head, seq_len, head_dim)
            k: (batch, n_kv_head, seq_len, head_dim)
        """
        return self.q_norm(q), self.k_norm(k)

### `SlidingWindowAttention`

This is the **key class** in the notebook. It handles both global and local attention using a single implementation that differs only in the mask.

#### Mask Construction -- The Core Idea

The mask is a 2D matrix of shape `(seq_len, seq_len)` where `mask[i, j] = 1` means "token at position `i` is allowed to attend to token at position `j`".

**Global mask** (standard causal): the familiar lower-triangular matrix. Token `i` can attend to all positions `0, 1, ..., i`. Built with `torch.tril`.

```
1 0 0 0 0
1 1 0 0 0
1 1 1 0 0
1 1 1 1 0
1 1 1 1 1
```

**Sliding window mask** (local causal): token `i` can attend to positions `max(0, i - window_size + 1)` through `i`. This is causal (no future tokens) AND local (no tokens more than `window_size` positions in the past).

For `window_size=3`:
```
1 0 0 0 0
1 1 0 0 0
1 1 1 0 0
0 1 1 1 0
0 0 1 1 1
```

Notice the diagonal band pattern. Token 3 can see tokens 1, 2, 3 but NOT token 0. The window slides forward with each position, giving the technique its name.

The mask is built with a simple loop:
```python
for i in range(seq_len):
    start = max(0, i - window_size + 1)
    mask[i, start:i + 1] = 1.0
```

Positions outside the mask receive `-inf` in the attention scores, so after softmax they contribute exactly zero.

#### The Full Forward Pass

The forward pass combines several techniques in order:
1. **Project** to Q, K, V (with GQA: fewer K, V heads than Q heads).
2. **Apply RoPE** to Q and K for position encoding.
3. **Apply QK-Norm** (if enabled) to stabilize attention scores.
4. **Expand K, V** for GQA by repeating KV heads to match Q heads.
5. **Compute attention** with the appropriate mask (global or sliding window).
6. **Project output** back to the model dimension.

In [ ]:
class SlidingWindowAttention(nn.Module):
    def __init__(
        self,
        config: SlidingWindowConfig,
        cos: torch.Tensor,
        sin: torch.Tensor,
        is_global: bool,
    ):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        assert config.n_head % config.n_kv_head == 0

        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_groups = config.n_head // config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.is_global = is_global

        # Q, K, V projections (GQA: K and V have fewer heads)
        self.W_q = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.W_k = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.W_v = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.W_o = nn.Linear(config.n_embd, config.n_embd, bias=False)

        self.dropout = nn.Dropout(config.dropout)

        # QK-Norm (optional)
        self.qk_norm = QKNorm(self.head_dim) if config.use_qk_norm else None

        # RoPE frequencies (shared, precomputed)
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        # Build the appropriate mask
        if is_global:
            # Standard causal mask (lower triangular)
            mask = torch.tril(torch.ones(config.seq_len, config.seq_len))
        else:
            # Sliding window causal mask
            # Position i can attend to positions max(0, i - window_size + 1) .. i
            mask = torch.zeros(config.seq_len, config.seq_len)
            for i in range(config.seq_len):
                start = max(0, i - config.window_size + 1)
                mask[i, start:i + 1] = 1.0

        self.register_buffer("mask", mask.view(1, 1, config.seq_len, config.seq_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # 1. Project to Q, K, V and reshape for multi-head attention
        q = self.W_q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        # 2. Apply RoPE to Q and K (not V -- values don't need position info)
        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        # 3. Apply QK-Norm if enabled
        if self.qk_norm is not None:
            q, k = self.qk_norm(q, k)

        # 4. Expand KV for GQA (repeat each KV head to match its group of Q heads)
        if self.n_groups > 1:
            k = k.repeat_interleave(self.n_groups, dim=1)
            v = v.repeat_interleave(self.n_groups, dim=1)

        # 5. Compute attention with the appropriate mask
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # 6. Weighted sum of values and project back
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_o(out)
        return out

### Mask Visualization

Let us build both mask types for a small sequence and display them side by side.
This makes the difference between global and local attention concrete and visual.

The left plot shows the standard causal mask: every token can attend to all previous tokens plus itself.
The right plot shows the sliding window mask: each token can only see a small neighborhood behind it.

Dark cells mean "allowed to attend". White cells mean "blocked".

In [ ]:
# Build masks for a small 16-token sequence with window_size=4
demo_seq_len = 16
demo_window = 4

# Global mask: standard lower-triangular causal
global_mask = torch.tril(torch.ones(demo_seq_len, demo_seq_len))

# Sliding window mask: causal AND within window
local_mask = torch.zeros(demo_seq_len, demo_seq_len)
for i in range(demo_seq_len):
    start = max(0, i - demo_window + 1)
    local_mask[i, start:i + 1] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(global_mask.numpy(), cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Global Mask (Full Causal)", fontsize=13)
axes[0].set_xlabel("Key position (j)")
axes[0].set_ylabel("Query position (i)")

axes[1].imshow(local_mask.numpy(), cmap="Blues", vmin=0, vmax=1)
axes[1].set_title(f"Sliding Window Mask (window={demo_window})", fontsize=13)
axes[1].set_xlabel("Key position (j)")
axes[1].set_ylabel("Query position (i)")

for ax in axes:
    ax.set_xticks(range(0, demo_seq_len, 2))
    ax.set_yticks(range(0, demo_seq_len, 2))

plt.tight_layout()
plt.show()

# Also print the patterns as text for environments without graphics
print("Global mask (16x16):")
for i in range(demo_seq_len):
    row = "".join(["#" if global_mask[i, j] == 1 else "." for j in range(demo_seq_len)])
    print(f"  pos {i:2d}: {row}")

print(f"\nSliding window mask (window={demo_window}):")
for i in range(demo_seq_len):
    row = "".join(["#" if local_mask[i, j] == 1 else "." for j in range(demo_seq_len)])
    print(f"  pos {i:2d}: {row}")

### `SlidingWindowBlock`

A transformer block combines two sublayers:
1. sliding window (or global) attention,
2. SwiGLU feed-forward MLP.

Both use **pre-norm** residual connections: normalize first, then apply the sublayer, then add back to the residual stream.

The block receives an `is_global` flag at construction time, which it passes through to `SlidingWindowAttention`. The block itself does not need to know whether it is global or local -- that distinction lives entirely in the mask.

In [ ]:
class SlidingWindowBlock(nn.Module):
    def __init__(
        self,
        config: SlidingWindowConfig,
        cos: torch.Tensor,
        sin: torch.Tensor,
        is_global: bool,
    ):
        super().__init__()
        self.is_global = is_global
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = SlidingWindowAttention(config, cos, sin, is_global)
        self.ln_2 = RMSNorm(config.n_embd)
        self.mlp = SwiGLU(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

## 6. Full `SlidingWindowModel`

This class assembles the complete language model. The key architectural choice happens in the constructor's loop:

```python
for i in range(config.n_layer):
    is_global = ((i + 1) % config.global_every == 0)
    self.blocks.append(SlidingWindowBlock(config, cos, sin, is_global))
```

With `n_layer=6` and `global_every=3`, the pattern is:
```
Layer 0: local   (sliding window)
Layer 1: local   (sliding window)
Layer 2: GLOBAL  (full attention)   <-- (0+1)=1, (1+1)=2, (2+1)=3 % 3 == 0
Layer 3: local   (sliding window)
Layer 4: local   (sliding window)
Layer 5: GLOBAL  (full attention)   <-- (5+1)=6 % 3 == 0
```

This means only 2 out of 6 layers pay the full O(n^2) cost. The other 4 layers use O(n * w) sliding window attention.

**No positional embedding table**: like Llama, this model relies entirely on RoPE for position encoding. RoPE frequencies are precomputed once and shared across all layers.

**Weight initialization**: residual-path projections (W_o in attention and w_down in SwiGLU) get smaller initial values, scaled by `1 / sqrt(2 * n_layer)`, to keep the residual stream stable as it accumulates contributions from many layers.

In [ ]:
class SlidingWindowModel(nn.Module):
    def __init__(self, config: SlidingWindowConfig):
        super().__init__()
        self.config = config

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)

        # Precompute RoPE frequencies (shared across all layers)
        head_dim = config.n_embd // config.n_head
        cos, sin = precompute_rope_freqs(head_dim, config.seq_len, config.rope_theta)

        # Build alternating local/global blocks
        self.blocks = nn.ModuleList()
        for i in range(config.n_layer):
            # Last layer in each group of global_every is global
            is_global = ((i + 1) % config.global_every == 0)
            self.blocks.append(SlidingWindowBlock(config, cos, sin, is_global))

        self.ln_f = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Initialize weights
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("W_o.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.seq_len, f"Sequence length {T} exceeds max {self.config.seq_len}"

        x = self.wte(idx)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        print(f"Total parameters: {total:,}")
        print(f"  Token embeddings: {self.wte.weight.numel():,}")
        for i, block in enumerate(self.blocks):
            block_params = sum(p.numel() for p in block.parameters())
            kind = "Global" if block.is_global else "Local"
            print(f"  Block {i} ({kind}):  {block_params:,}")
        print(f"  Final norm: {sum(p.numel() for p in self.ln_f.parameters()):,}")
        print(f"  LM head:   {self.lm_head.weight.numel():,}")
        return total

## 7. Model Smoke Test

Before training anything, we run a quick confidence check. This verifies:
1. The alternating local/global pattern is correct.
2. The local mask blocks distant tokens (position 255 cannot attend to position 0).
3. The global mask allows full causal attention (position 255 CAN attend to position 0).
4. QK-Norm is present and active.
5. The logit shapes are correct.
6. The random-initialization loss is near `log(256) = 5.54`.
7. Gradients flow through the entire network.

Teaching note: running these checks costs almost nothing and catches many integration bugs before they waste a training run.

In [ ]:
model = SlidingWindowModel(config)
model.count_parameters()

# 1. Verify the alternating pattern
print("\nAttention pattern:")
pattern = []
for i, block in enumerate(model.blocks):
    kind = "GLOBAL" if block.is_global else "local"
    pattern.append(kind)
    print(f"  Block {i}: {kind}")

assert pattern == ["local", "local", "GLOBAL", "local", "local", "GLOBAL"], \
    f"Unexpected pattern: {pattern}"
print("Alternating pattern verified.")

# 2. Verify sliding window mask blocks distant tokens
local_block = model.blocks[0]
assert not local_block.is_global
local_mask = local_block.attn.mask[0, 0]
assert local_mask[-1, 0] == 0, "Local mask should block position 255 -> position 0"
print(f"\nSliding window: position {config.seq_len-1} correctly BLOCKED from position 0")

# 3. Verify global mask allows full attention
global_block = model.blocks[2]
assert global_block.is_global
global_mask_val = global_block.attn.mask[0, 0]
assert global_mask_val[-1, 0] == 1, "Global mask should allow position 255 -> position 0"
print(f"Global attention: position {config.seq_len-1} correctly ATTENDS to position 0")

# 4. Verify QK-Norm is present
assert local_block.attn.qk_norm is not None, "QK-Norm should be enabled"
print(f"QK-Norm: enabled")

# 5. Forward pass shape check
idx = torch.randint(0, config.vocab_size, (2, 32))
logits, _ = model(idx)
print(f"\nLogits shape: {logits.shape}")
assert logits.shape == (2, 32, config.vocab_size)

# 6. Loss sanity check
targets = torch.randint(0, config.vocab_size, (2, 32))
logits, loss = model(idx, targets)
print(f"Random-init loss: {loss.item():.4f}")
print(f"Expected rough baseline: {math.log(config.vocab_size):.4f}")
assert abs(loss.item() - math.log(config.vocab_size)) < 1.0

# 7. Gradient flow
loss.backward()
grad_count = sum(1 for p in model.parameters() if p.grad is not None)
print(f"Parameters with gradients: {grad_count}")
print("\nAll smoke tests passed!")

## 8. Data Pipeline

A language model learns from one very long stream of tokens.
Our job is to turn that stream into many smaller `(input, target)` examples.

In this notebook we use the simplest tokenizer possible: raw UTF-8 bytes.
That keeps the modeling ideas front and center.

### Dataset Paths and Constants

Notebooks are often run from different working directories.
This cell finds the project root in a way that works both when the notebook is opened from the repository root and when it is opened from the `notebook/` folder.

We also define the Tiny Shakespeare download URL here so later functions can stay focused on behavior rather than configuration.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "model.py").exists() and (PROJECT_ROOT.parent / "model.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "sliding_window"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

### `encode` and `decode`

`encode` converts text into integer token IDs using raw UTF-8 bytes (range 0..255).
`decode` reverses the process. The `errors="replace"` flag keeps decoding robust even if the model emits invalid byte combinations.

In [ ]:
def encode(text: str) -> list[int]:
    return list(text.encode("utf-8"))

def decode(tokens: list[int]) -> str:
    return bytes(tokens).decode("utf-8", errors="replace")

### `download_shakespeare`, `get_batch`, `prepare_data`

These three functions handle the complete data pipeline:
- `download_shakespeare`: downloads and caches the Tiny Shakespeare dataset.
- `get_batch`: randomly samples `(input, target)` pairs from the token stream. The target is the input shifted by one position.
- `prepare_data`: ties it all together -- download, encode, tensorize, split into train/val.

In [ ]:
def download_shakespeare() -> str:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    filepath = DATA_DIR / "shakespeare.txt"

    if not filepath.exists():
        print(f"Downloading tiny Shakespeare to {filepath}...")
        response = requests.get(SHAKESPEARE_URL)
        response.raise_for_status()
        filepath.write_text(response.text)
        print(f"Downloaded {len(response.text):,} characters.")

    return filepath.read_text()


def get_batch(data: torch.Tensor, batch_size: int, seq_len: int, device: str = "cpu"):
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in starts])
    y = torch.stack([data[i + 1 : i + 1 + seq_len] for i in starts])
    return x.to(device), y.to(device)


def prepare_data(val_fraction: float = 0.1):
    text = download_shakespeare()
    tokens = encode(text)
    data = torch.tensor(tokens, dtype=torch.long)
    split_idx = int(len(data) * (1 - val_fraction))
    train_data = data[:split_idx]
    val_data = data[split_idx:]
    return train_data, val_data

## 9. Data Smoke Test

We test three things here:
1. Tokenizer round-trip correctness.
2. Dataset size after preparation.
3. Shift-by-one behavior in the training batch.

When teaching, these checks are valuable because many training bugs begin in data handling, not in the model itself.

In [ ]:
sample_text = "Hello, World!"
sample_tokens = encode(sample_text)
round_trip = decode(sample_tokens)
print(sample_tokens)
print(round_trip)
assert round_trip == sample_text

train_data, val_data = prepare_data()
print(f"Total tokens: {len(train_data) + len(val_data):,}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")

x, y = get_batch(train_data, batch_size=4, seq_len=32)
print(f"Batch shapes: x={x.shape}, y={y.shape}")
print(f"First input preview:  {decode(x[0].tolist())!r}")
print(f"First target preview: {decode(y[0].tolist())!r}")
assert torch.equal(x[0, 1:], y[0, :-1])

## 10. Training Utilities

The next few functions make the training loop easier to read.
Splitting them out is good software design and good teaching design: each helper introduces one concept at a time.

### `get_lr`

This function implements a two-stage learning-rate schedule:
1. **Warmup**: start small so early updates are stable.
2. **Cosine decay**: gradually lower the learning rate so the model can settle into a better solution.

In [ ]:
def get_lr(step: int, max_lr: float, min_lr: float, warmup_steps: int, total_steps: int) -> float:
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps

    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + (max_lr - min_lr) * cosine

### `estimate_loss`

Training loss on a single minibatch is noisy.
`estimate_loss` gives a more trustworthy picture by averaging over several fresh batches from both the training split and the validation split.

In [ ]:
@torch.no_grad()
def estimate_loss(model, train_data, val_data, batch_size, seq_len, device, eval_iters=20):
    model.eval()
    results = {}
    for name, data in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(data, batch_size, seq_len, device)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[name] = sum(losses) / len(losses)
    model.train()
    return results

### `save_checkpoint` and `load_checkpoint`

A checkpoint captures the current training state so you can pause work, resume later, or keep the best-performing model. We save the model weights, optimizer state, config, and current step.

`load_checkpoint` rebuilds a `SlidingWindowModel` from the saved config and restores the learned weights.

In [ ]:
def save_checkpoint(model, config, optimizer, step, path):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "step": step,
        },
        path,
    )


def load_checkpoint(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint["config"]
    model = SlidingWindowModel(config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, config, checkpoint["step"]

### `get_device` and `sanity_check`

`get_device` picks the best available accelerator (CUDA > MPS > CPU).

`sanity_check` overfits a single fixed batch. If the loss does not drop near zero, there is a fundamental bug in the model, data pipeline, or optimization loop. This is one of the most useful debugging tricks in deep learning.

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def sanity_check(config, device):
    print("=" * 60)
    print("SANITY CHECK: overfitting one batch")
    print("=" * 60)

    model = SlidingWindowModel(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    train_data, _ = prepare_data()
    x, y = get_batch(train_data, batch_size=4, seq_len=config.seq_len, device=device)

    for step in range(500):
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            print(f"step {step:4d} | loss {loss.item():.4f}")

    final_loss = loss.item()
    print(f"Final loss: {final_loss:.4f}")
    return final_loss < 0.1

### `train`

This is the full training loop. It combines every previous idea: data loading, model creation, optimizer setup, learning-rate scheduling, gradient computation, gradient clipping, validation checks, and checkpoint saving.

As you read it, notice the rhythm of training:
1. Choose a learning rate for the current step.
2. Sample a batch.
3. Run forward and backward passes.
4. Update the weights.
5. Occasionally evaluate and save.

In [ ]:
def train(
    config,
    device,
    max_steps=2000,
    batch_size=32,
    max_lr=3e-3,
    min_lr=3e-4,
    warmup_steps=100,
    eval_interval=100,
    save_interval=500,
    checkpoint_dir=CHECKPOINT_DIR,
):
    print("=" * 60)
    print("TRAINING")
    print("=" * 60)
    print(f"Device: {device}")
    print(f"Config: n_embd={config.n_embd}, n_head={config.n_head}, n_layer={config.n_layer}")
    print(f"  window_size={config.window_size}, global_every={config.global_every}, qk_norm={config.use_qk_norm}")
    print(f"Steps: {max_steps}, Batch size: {batch_size}, Seq len: {config.seq_len}")
    print(f"LR: {max_lr} -> {min_lr} (warmup {warmup_steps} steps)")
    print()

    train_data, val_data = prepare_data()
    model = SlidingWindowModel(config).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")

    decay_params = [p for p in model.parameters() if p.dim() >= 2]
    nodecay_params = [p for p in model.parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": 0.1},
            {"params": nodecay_params, "weight_decay": 0.0},
        ],
        lr=max_lr,
        betas=(0.9, 0.95),
    )

    use_amp = device in ("cuda", "mps")
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    amp_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

    os.makedirs(checkpoint_dir, exist_ok=True)
    best_val_loss = float("inf")
    t0 = time.time()

    for step in range(max_steps):
        lr = get_lr(step, max_lr, min_lr, warmup_steps, max_steps)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        x, y = get_batch(train_data, batch_size, config.seq_len, device)

        if use_amp:
            with torch.autocast(device_type=device, dtype=amp_dtype):
                _, loss = model(x, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            _, loss = model(x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)

        if step % 10 == 0:
            elapsed = time.time() - t0
            tokens_per_sec = (step + 1) * batch_size * config.seq_len / elapsed if elapsed > 0 else 0
            print(f"step {step:5d} | loss {loss.item():.4f} | lr {lr:.2e} | {tokens_per_sec:.0f} tok/s")

        if step > 0 and step % eval_interval == 0:
            losses = estimate_loss(model, train_data, val_data, batch_size, config.seq_len, device)
            print(f"eval | train {losses['train']:.4f} | val {losses['val']:.4f}")
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                save_checkpoint(model, config, optimizer, step, Path(checkpoint_dir) / "best.pt")
                print("saved new best checkpoint")

        if step > 0 and step % save_interval == 0:
            save_checkpoint(model, config, optimizer, step, Path(checkpoint_dir) / "latest.pt")

    save_checkpoint(model, config, optimizer, max_steps, Path(checkpoint_dir) / "final.pt")
    print(f"Training complete. Best val loss: {best_val_loss:.4f}")
    return model

## 11. Lightweight Pipeline Preview

Instead of launching a full training run right away, this cell does a quick end-to-end pass:
- choose a device,
- build the config and model,
- load data,
- sample one batch,
- compute one forward pass.

This is a great habit in real projects because it catches integration problems early while staying fast enough for iteration.

In [ ]:
device = get_device()
config = SlidingWindowConfig()
model = SlidingWindowModel(config).to(device)
train_data, val_data = prepare_data()
x, y = get_batch(train_data, batch_size=2, seq_len=32, device=device)
logits, loss = model(x, y)

print(f"Selected device: {device}")
print(f"Input batch shape:  {x.shape}")
print(f"Target batch shape: {y.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"One-batch loss:     {loss.item():.4f}")

## 12. Text Generation

Training teaches the model a probability distribution over the next token.
Generation turns that distribution into actual text by repeating the same loop:
1. Feed the current context into the model.
2. Read the logits for the last position.
3. Choose the next token (with temperature and top-k sampling).
4. Append it to the context.
5. Repeat.

Temperature and top-k are two simple ways to control the sampling behavior:
- **Temperature**: lower values make the model more conservative; higher values make it more random.
- **Top-k**: restricts sampling to the most plausible candidates, reducing wild mistakes.

In [ ]:
@torch.no_grad()
def generate(model, prompt_tokens: list[int], max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 50, device: str = "cpu") -> list[int]:
    model.eval()
    seq_len = model.config.seq_len
    tokens = torch.tensor([prompt_tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx = tokens[:, -seq_len:]
        logits, _ = model(idx)
        logits = logits[:, -1, :]

        if temperature == 0:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_values[:, -1:]] = float("-inf")
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat([tokens, next_token], dim=1)

    return tokens[0].tolist()

## 13. Optional Checkpoint Demo

This cell tries to load a saved checkpoint and generate text.
If no checkpoint exists yet, it prints a friendly message instead of failing.

That makes the notebook safe to run top-to-bottom in a fresh clone while still showing how inference works once you have trained the model.

In [ ]:
checkpoint_path = CHECKPOINT_DIR / "best.pt"

if checkpoint_path.exists():
    demo_model, demo_config, step = load_checkpoint(checkpoint_path, device)
    prompt_tokens = encode("ROMEO:")
    output_tokens = generate(
        demo_model,
        prompt_tokens,
        max_new_tokens=200,
        temperature=0.8,
        top_k=50,
        device=device,
    )
    print(f"Loaded checkpoint from step {step}")
    print(decode(output_tokens))
else:
    print(f"No checkpoint found at {checkpoint_path}.")
    print("Run a training cell first if you want to try generation from learned weights.")

## 14. Exercises

Try these experiments to deepen your understanding of sliding window attention.

### Exercise 1: Change the Window Size

Modify `window_size` and observe the effect on the mask and training.

```python
# Try a very small window (e.g., 16) and a very large window (e.g., 128)
small_window_config = SlidingWindowConfig(window_size=16)
large_window_config = SlidingWindowConfig(window_size=128)
```

Questions to think about:
- How does a smaller window affect the model's ability to learn long-range patterns?
- At what point does increasing the window size stop helping?
- What happens when `window_size >= seq_len`? (Hint: the local mask becomes identical to the global mask.)

### Exercise 2: Change the Local:Global Ratio

Modify `global_every` to change how often global layers appear.

```python
# Gemma 3 style: 5:1 local:global
gemma_config = SlidingWindowConfig(global_every=6)

# All global (standard transformer)
all_global_config = SlidingWindowConfig(global_every=1)

# All local (Mistral v1 style)
all_local_config = SlidingWindowConfig(global_every=999)
```

Questions to think about:
- With `global_every=1`, every layer is global. How does this compare to a standard Llama model?
- With very rare global layers, does the model still learn long-range dependencies through the effective receptive field?
- What is the compute trade-off between more global layers and better long-range modeling?

### Exercise 3: Disable QK-Norm

Turn off QK-Norm and compare training stability.

```python
no_qknorm_config = SlidingWindowConfig(use_qk_norm=False)
```

Questions to think about:
- At this small scale, does QK-Norm make a noticeable difference?
- Monitor the attention logit magnitudes: do they grow larger without QK-Norm?
- Why might QK-Norm matter more at larger scale (e.g., 7B parameters)?

In [ ]:
# Exercise scaffold: compare window sizes
# Uncomment and run to see the mask difference

# small_config = SlidingWindowConfig(window_size=16)
# small_model = SlidingWindowModel(small_config)
# small_mask = small_model.blocks[0].attn.mask[0, 0]
# print("Small window (16) - last row attends to positions:")
# print(f"  {torch.where(small_mask[-1] == 1)[0].tolist()}")
# print(f"  That's {small_mask[-1].sum().int().item()} positions out of {config.seq_len}")
#
# large_config = SlidingWindowConfig(window_size=128)
# large_model = SlidingWindowModel(large_config)
# large_mask = large_model.blocks[0].attn.mask[0, 0]
# print(f"\nLarge window (128) - last row attends to positions:")
# print(f"  {torch.where(large_mask[-1] == 1)[0].tolist()[:10]}... (showing first 10)")
# print(f"  That's {large_mask[-1].sum().int().item()} positions out of {config.seq_len}")

## 15. Common Pitfalls and Next Steps

### Common Pitfalls
- **Running cells out of order**: later cells depend on earlier imports and class definitions.
- **Confusing `window_size` with `seq_len`**: `window_size` controls how far back each local layer can look. `seq_len` is the maximum total context the model can process.
- **Forgetting that global layers exist**: even with sliding window, this model has full-attention layers. Do not assume all attention is local.
- **Expecting dramatic speed differences at small scale**: with `seq_len=256` and `window_size=64`, the computational savings from sliding window are modest. The real payoff comes at 4K+ sequence lengths where full attention becomes prohibitively expensive.
- **Judging learning from one noisy batch**: use `estimate_loss` instead of trusting a single minibatch.
- **Expecting fast training on CPU**: even a tiny transformer is much more pleasant on GPU.

### How This Connects to Production Models

The architecture in this notebook is a simplified version of what runs at scale:

| Feature | This Notebook | Gemma 3 27B | OLMo 3 |
|---------|--------------|-------------|---------|
| Window size | 64 | 4,096 | 4,096 |
| Local:Global ratio | 2:1 | 5:1 | 3:1 |
| QK-Norm | Yes | Yes | Yes |
| GQA | Yes | Yes | Yes |
| RoPE | Yes | Yes | Yes |

### Next Steps
- Compare training curves between all-global, all-local, and mixed architectures.
- Implement a longer sequence length and measure the actual compute savings of sliding window.
- Add KV-cache for efficient autoregressive generation.
- Explore different window sizes for different layers (some models use increasing window sizes in deeper layers).
- Read the Gemma 3 technical report to see how these ideas scale to 27 billion parameters.